In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from conus_biomass import dir_info

In [ ]:
dir_processed_model_output = dir_info.dir_model_output[:-1] + "_processed/"

In [ ]:
for i in np.arange(0, 50):
    model_suffix = f"_{i:04d}"
    # print(model_suffix)
    df = pd.read_csv(dir_processed_model_output + "MMTC_our_study" + model_suffix + ".csv")
    df = df.drop(columns=["Unnamed: 0"])
    df["model_suffix"] = model_suffix
    if i == 0:
        df_all = df
    else:
        df_all = pd.concat([df_all, df])

In [ ]:
df_all_westwide = df_all.groupby(["year", "model_suffix"]).sum()["live_biomass_MMT"].reset_index()

In [ ]:
# Get the 2005 baseline for each model_suffix
baseline = df_all_westwide[df_all_westwide["year"] == 2005].set_index("model_suffix")[
    "live_biomass_MMT"
]

# Subtract the matching 2005 value based on model_suffix
df_all_westwide["live_biomass_MMT_delta"] = df_all_westwide["live_biomass_MMT"] - df_all_westwide[
    "model_suffix"
].map(baseline)

df_all_westwide["live_biomass_MMT_frac_delta"] = df_all_westwide[
    "live_biomass_MMT"
] / df_all_westwide["model_suffix"].map(baseline)

In [ ]:
non_CONUS = ["Alaska", "Hawaii", "U.S. Territories", "Puerto Rico"]
western_states = [
    "California",
    "Washington",
    "Oregon",
    "Idaho",
    "Montana",
    "Arizona",
    "Colorado",
    "New Mexico",
    "Utah",
    "Wyoming",
    "Nevada",
]
years = np.arange(2005, 2023)

FRF_state = pd.read_csv(dir_info.dir_walters + "FRF_stock_by_State.csv")

FRF_state_agb = FRF_state[FRF_state["Carbon Pools"] == "Aboveground Biomass"].drop(
    columns=["Carbon Pools"]
)
FRF_state_conus = FRF_state_agb[~FRF_state_agb["State"].isin(non_CONUS)]

CONUS_stocks = FRF_state_conus.drop(columns=["State"]).sum()
CONUS_stocks.index = CONUS_stocks.index.astype(int)

FRF_state_western = FRF_state_conus[FRF_state_conus["State"].isin(western_states)]
western_stocks = FRF_state_western.drop(columns=["State"]).sum()
western_stocks.index = western_stocks.index.astype(int)

FRF_state_eastern = FRF_state_conus[~FRF_state_conus["State"].isin(western_states)]
eastern_stocks = FRF_state_eastern.drop(columns=["State"]).sum()
eastern_stocks.index = eastern_stocks.index.astype(int)

western_stocks_delta = western_stocks - (western_stocks[western_stocks.index == years[0]].values)
eastern_stocks_delta = eastern_stocks - (eastern_stocks[eastern_stocks.index == years[0]].values)
CONUS_stocks_delta = CONUS_stocks - (CONUS_stocks[CONUS_stocks.index == years[0]].values)

USFS_years = CONUS_stocks.index

In [ ]:
biomass_mean_west = df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).median()
biomass_min_west = (
    df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).quantile(0.025)
)
biomass_max_west = (
    df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).quantile(0.975)
)

biomass_mean_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).median()
)
biomass_min_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).quantile(0.025)
)
biomass_max_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).quantile(0.975)
)

In [ ]:
plt.figure(figsize=(20, 6))
plt.subplot(1, 3, 1)
for i in np.arange(0, 30):
    model_suffix = f"_{i:04d}"
    df_onemodel = df_all_westwide[df_all_westwide["model_suffix"] == model_suffix]
    plt.plot(df_onemodel["year"], df_onemodel["live_biomass_MMT"], "-", alpha=0.1, color="black")

plt.subplot(1, 3, 2)
for i in np.arange(0, 30):
    model_suffix = f"_{i:04d}"
    df_onemodel = df_all_westwide[df_all_westwide["model_suffix"] == model_suffix]
    plt.plot(
        df_onemodel["year"], df_onemodel["live_biomass_MMT_delta"], "-", alpha=0.1, color="black"
    )

plt.subplot(1, 3, 3)
for i in np.arange(0, 30):
    model_suffix = f"_{i:04d}"
    df_onemodel = df_all_westwide[df_all_westwide["model_suffix"] == model_suffix]
    plt.plot(
        df_onemodel["year"],
        df_onemodel["live_biomass_MMT_frac_delta"],
        "-",
        alpha=0.1,
        color="black",
    )

# Absolute biomass stock

In [ ]:
plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(8, 6))

plt.plot(USFS_years, western_stocks, "-", linewidth=5, color="gray", label="USFS/EPA Estimate")
plt.plot(
    years,
    (np.array(biomass_mean_west)),
    ".-",
    linewidth=5,
    color="C0",
    label="This Study",
)
plt.fill_between(
    years,
    (np.array(biomass_min_west)),
    (np.array(biomass_max_west)),
    color="C0",
    alpha=0.4,
    edgecolor=None,
)

plt.title("Western US")
plt.ylabel("Live AGB (MMT C)")
ax.spines[["right", "top"]].set_visible(False)
plt.legend(frameon=False)
ax.set_xlim([2005, 2023])
plt.axvspan(2005, 2015, color="lightgray", alpha=0.3)

# Difference over time for each ensemble member

In [ ]:
plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(8, 6))

plt.plot(
    USFS_years, western_stocks_delta, "-", linewidth=5, color="gray", label="USFS/EPA Estimate"
)
plt.plot(
    years,
    (np.array(biomass_mean_west_delta)),
    ".-",
    linewidth=5,
    color="C0",
    label="This Study",
)
plt.fill_between(
    years,
    (np.array(biomass_min_west_delta)),
    (np.array(biomass_max_west_delta)),
    color="C0",
    alpha=0.4,
    edgecolor=None,
)

plt.title("Western US")
plt.ylabel("Cumulative Change in Live AGB (MMT C)")
ax.spines[["right", "top"]].set_visible(False)
plt.legend(frameon=False)
ax.set_xlim([2005, 2023])
plt.axhline(y=0, linestyle=":", color="gray", linewidth=1)
plt.axvspan(2005, 2015, color="lightgray", alpha=0.3)

# Difference from median estimate

In [ ]:
plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(8, 6))

plt.plot(
    years,
    (np.array(biomass_mean_west)) - (np.array(biomass_mean_west))[0],
    ".-",
    linewidth=5,
    color="C0",
    label="This Study",
)
plt.fill_between(
    years,
    (np.array(biomass_min_west)) - (np.array(biomass_mean_west))[0],
    (np.array(biomass_max_west)) - (np.array(biomass_mean_west))[0],
    color="C0",
    alpha=0.4,
    edgecolor=None,
)
plt.plot(
    USFS_years, western_stocks_delta, "-", linewidth=5, color="gray", label="USFS/EPA Estimate"
)

plt.title("Western US")
plt.ylabel("Cumulative Change in Live AGB (MMT C)")
ax.spines[["right", "top"]].set_visible(False)
plt.legend(frameon=False)
ax.set_xlim([2005, 2023])
plt.axhline(y=0, linestyle=":", color="gray", linewidth=1)
plt.axvspan(2005, 2015, color="lightgray", alpha=0.3)